# Import Dataset

In [ ]:
import pandas as pd

df = pd.read_excel('Program_target.xlsx')
df.head()

,City/Municipality,Poverty_Incidence,Current_Target
0,CALOOCAN CITY,0.304766,7774
1,CITY OF LAS PIÑAS,0.050549,1390
2,CITY OF MAKATI,0.049783,1144
3,CITY OF MALABON,0.053606,2458
4,CITY OF MANDALUYONG,0.016361,939


In [ ]:
# --- National Proposed Physical Target ---
T_nat = 4_400_000

# Step 1: Compute poverty share and ideal allocation
df['Poverty_Share'] = df['Poverty_Incidence'] / df['Poverty_Incidence'].sum()
df['Ideal_Target'] = df['Poverty_Share'] * T_nat

# Step 2: Identify locked targets
df['Final_Target'] = df['Current_Target']  # Start from current (locked) values

# Step 3: Remaining allocation pool
locked_total = df['Current_Target'].sum()
available = T_nat - locked_total

# Step 4: Residual need for LGUs (based on poverty incidence)
df['Residual_Need'] = df['Ideal_Target'] - df['Current_Target']
df['Residual_Need'] = df['Residual_Need'].clip(lower=0)
total_residual = df['Residual_Need'].sum()

# Step 5: Allocate remaining slots proportionally
if total_residual > 0 and available > 0:
    df['Residual_Share'] = df['Residual_Need'] / total_residual
    df['Additional_Allocation'] = df['Residual_Share'] * available
    df['Final_Target'] += df['Additional_Allocation'].round()
else:
    df['Additional_Allocation'] = 0

# Step 6: Final rounding correction (so total == 4,400,000)
diff = int(T_nat - df['Final_Target'].sum())

if diff != 0:
    eligible = df[df['Residual_Need'] > 0]
    if not eligible.empty:
        idx = eligible['Final_Target'].idxmax()
        df.at[idx, 'Final_Target'] += diff

# Step 7: Enforce non-negative targets (failsafe)
df['Final_Target'] = df[['Final_Target', 'Current_Target']].max(axis=1)
df['Final_Target'] = df['Final_Target'].clip(lower=0)

# Final check
final_total = int(df['Final_Target'].sum())
print("✅ Final total target =", final_total)
print(df[['City/Municipality', 'Current_Target', 'Poverty_Incidence', 'Ideal_Target', 'Final_Target']].head())



✅ Final total target = 4400000
     City/Municipality  Current_Target  Poverty_Incidence  Ideal_Target  \
0        CALOOCAN CITY            7774           0.304766  13409.719824   
1    CITY OF LAS PIÑAS            1390           0.050549   2224.170664   
2       CITY OF MAKATI            1144           0.049783   2190.471108   
3      CITY OF MALABON            2458           0.053606   2358.650612   
4  CITY OF MANDALUYONG             939           0.016361    719.880752   

   Final_Target  
0       13394.0  
1        2222.0  
2        2188.0  
3        2458.0  
4         939.0  
